In [1]:
%load_ext autoreload
%autoreload 2

# Tabel verb + da andmete kogumine


**Ülesande püstitus**
    
Kõik laused, kus esineb tabelis [list_da.csv](./lists/101.list_da.csv) olev verb ja verbil on otsene alluv deprel=xcomp, feats sisaldab inf

**Tulemus** 

Tabel veergudega:
1. leitud lause, 
2. milline tabelis olevates verbidest seal esineb (algvormis), 
3. da-infinitiivi vormis oleva verbi lemma, 
4. keeletase.
   

In [2]:
import pandas as pd
from datetime import datetime
from notebook_context import corpus_reader, LISTS_FOLDER

date_time = datetime.now().strftime("%Y%m%d-%H%M%S")


DA_VERBS_LIST =   "./lists/101.list_da.csv"
RESULTS_FILE= LISTS_FOLDER / f"results/da_loend_xcomp_{date_time}.csv"

In [3]:
%%time

# verbid etteantud nimekirjast
df_verbs = pd.read_csv(DA_VERBS_LIST)
my_verbs = list(df_verbs['lemma'].unique())

CPU times: user 2.3 ms, sys: 4.41 ms, total: 6.7 ms
Wall time: 5.94 ms


In [4]:
my_verbs

['saama',
 'suutma',
 'jaksama',
 'jõudma',
 'nägema',
 'oskama',
 'teadma',
 'mõistma',
 'tohtima',
 'võima',
 'tahtma',
 'kavatsema',
 'plaanima',
 'otsustama',
 'lootma',
 'soovima',
 'igatsema',
 'ihkama',
 'maldama',
 'kärsima',
 'läbema',
 'unistama',
 'ootama',
 'himustama',
 'taotlema',
 'ilgema',
 'sügelema',
 'kibelema',
 'janunema',
 'kaaluma',
 'kavandama',
 'kokku leppima',
 'mõtlema',
 'plaanitsema',
 'planeerima',
 'sihtima',
 'märkama',
 'taipama',
 'unustama',
 'kartma',
 'häbenema',
 'armastama',
 'eelistama',
 'julgema',
 'söandama',
 'tihkama',
 'riskima',
 'usaldama',
 'riskeerima',
 'uskuma',
 'suvatsema',
 'viitsima',
 'paljuks pidama',
 'raatsima',
 'täima',
 'vaevaks võtma',
 'pelgama',
 'põlgama',
 'tõrkuma',
 'kõhklema',
 'pruukima',
 'tarvitsema',
 'lubama',
 'ähvardama',
 'tõotama',
 'vanduma',
 'proovima',
 'püüdma',
 'katsuma',
 'üritama',
 'tavatsema',
 'harrastama',
 'väärima',
 'aitama',
 'ette_panema',
 'hõlbustama',
 'keelama',
 'käskima',
 'laskma',

In [5]:
%%time


collected_data = []
count = 0
for collection_id, graph in corpus_reader.get_sentences():
    # matrix for node distances
    dpath = graph.get_distances_matrix()
    
    # verb nodes
    verb_nodes = [v for v in graph.get_nodes_by_attributes(attrname="POS", attrvalue="VERB") if graph.nodes[v]["lemma"] in my_verbs]
    if not len(verb_nodes): continue
    
    # xcomp
    xcomp_nodes = graph.get_nodes_by_attributes(attrname="deprel", attrvalue="xcomp")
   
    if not len(xcomp_nodes): continue
    
    for verb in verb_nodes:
        # childnodes
        kids = [k for k in dpath[verb] if dpath[verb][k] == 1]
        for xcomp in xcomp_nodes:
            if xcomp not in kids:
                continue
            if not graph.nodes[xcomp]["feats"] or "VerbForm" not in graph.nodes[xcomp]["feats"].keys() or not graph.nodes[xcomp]["feats"]["VerbForm"] == 'Inf':
                continue
            
            #graph.draw_graph2(highlight=[verb, xcomp])
            d = {
                'id':  graph.get_metadata('row_id'),
                'sentence':  graph.get_metadata('text'),
                'verb':  graph.nodes[verb]["lemma"],
                'xcomp':  graph.nodes[xcomp]["lemma"],
                'level':  graph.get_metadata('sent_level'),
                'sub': " ".join(
                            [graph.nodes[n]["form"] for n in sorted([verb] + kids)]
                        ),
            }
            
            collected_data.append(d)


../data/vrt-with-meta-corpus-02-06-25_ordered.vrt
CPU times: user 31.6 s, sys: 153 ms, total: 31.8 s
Wall time: 31.8 s


In [6]:
df = pd.DataFrame.from_dict(collected_data)
df.to_csv(RESULTS_FILE, index=None)
df.head(10)

,id,sentence,verb,xcomp,level,sub
0,None,None,tahtma,lugema,None,ja tahtis lugeda
1,None,None,saama,teadma,None,", et teada saada palju"
2,None,None,tahtma,teadma,None,Ta tahtis teada .
3,None,None,tahtma,teadma,None,", et ta tahtis teada ilusad"
4,None,None,tahtma,lugema,None,", sest ta tahtis lugeda"
5,None,None,tahtma,teadma,None,Ta tahtis teada palju ilusad .
6,None,None,saama,teadma,None,", et teada saada palju"
7,None,None,tahtma,teadma,None,Keku tahtis teada .
8,None,None,tahtma,kittuma,None,Ullu tahtis kittuda .
9,None,None,tahtma,teadma,None,", sest ta tahtis teada"
